# Ìmọ̀lára · `e6a_afriberta_nodiacritics`

E6a: AfriBERTa (E4) trained on fully undiacritised text

Runs `python -m src.train_transformer --config configs/e6a_afriberta_nodiacritics.yaml` for seeds 42, 43, 44 on a Kaggle T4 GPU.

* **Code:** [Hassan-Adelani-Luqman/imolara-yoruba-sentiment](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment) at commit [`fa65056`](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment/tree/fa6505695769169eb6ad1e96e4024c9ebe44f4f6), cloned below; nothing is edited inside this notebook.
* **Data:** AfriSenti Yoruba (raw TSVs, pinned commit), downloaded by `src/data.py`.
* **Outputs:** `/kaggle/working/outputs/seed*/` (metrics, dev predictions, run info), downloaded to `results/kaggle/e6a_afriberta_nodiacritics/` in the repo.
* **Reproduce:** *Copy & Edit*, set Accelerator = GPU T4 and Internet = on, then *Run all*.

Generated by `scripts/kaggle_run.py --notebook`.

## 1 · Setup: clone the repo at the pinned commit, install extras, check the GPU

In [1]:
import glob, json, os, subprocess, sys

REPO, COMMIT = "https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment.git", "fa6505695769169eb6ad1e96e4024c9ebe44f4f6"
CODE, OUT = "/tmp/imolara", "/kaggle/working/outputs"
CONFIG, SEEDS = "configs/e6a_afriberta_nodiacritics.yaml", [42, 43, 44]

subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements-kaggle.txt"], check=True)
# Kaggle ships torchao 0.10, which peft >= 0.20 rejects at import time; we don't use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/env.txt", "w") as f:            # exact package versions, for the report
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)

env = {**os.environ, "IMOLARA_GIT_COMMIT": COMMIT[:7], "PYTHONUNBUFFERED": "1"}
emb = glob.glob("/kaggle/input/**/w2v_yo_300.kv", recursive=True)
if emb:                                            # attached imolara-embeddings dataset (RNN runs)
    env["IMOLARA_EMB_DIR"] = os.path.dirname(emb[0])
    print("embeddings:", env["IMOLARA_EMB_DIR"])

GPU: Tesla T4 | torch 2.11.0+cu128


## 2 · Experiment configuration

In [2]:
print(open(f"{CODE}/{CONFIG}").read())

# E6a: AfriBERTa (E4) trained on fully undiacritised text
model_name: castorini/afriberta_large
learning_rate: 3.0e-5
batch_size: 32
epochs: 10
early_stopping_patience: 3
eval_diacritics: [original, no_tones, no_diacritics]
train_diacritics: no_diacritics



## 3 · Training: one process per seed, one GPU per process

Kaggle's T4 machine has two GPUs. Each seed trains in its own process pinned to one GPU (`CUDA_VISIBLE_DEVICES`), so two seeds run at once and `batch_size` in the config is the real batch (no DataParallel). Log lines are prefixed with their seed. The best epoch is chosen on *clean* dev macro-F1 (dev tweets that do not duplicate a training tweet).

In [3]:
import queue, threading

N_GPU = max(1, torch.cuda.device_count())
print(f"{N_GPU} GPU(s) -> up to {N_GPU} seeds in parallel", flush=True)
NOISE = ("Warning", "warn(", "it/s]", "Loading weights")

def run(seed, gpu):
    cmd = [sys.executable, "-m", "src.train_transformer", "--config", CONFIG, "--seed", str(seed),
           "--output_dir", f"{OUT}/seed{seed}", *[]]
    proc = subprocess.Popen(cmd, cwd=CODE, env={**env, "CUDA_VISIBLE_DEVICES": str(gpu)},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if not any(n in line for n in NOISE):
            print(f"[seed {seed} | gpu {gpu}] {line}", end="", flush=True)
    return proc.wait()

def run_all(seeds):
    todo, codes = queue.Queue(), {}
    for s in seeds:
        todo.put(s)
    def worker(gpu):
        while True:
            try:
                seed = todo.get_nowait()
            except queue.Empty:
                return
            codes[seed] = run(seed, gpu)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    failed = [s for s, rc in codes.items() if rc != 0]
    assert not failed, f"seeds failed: {failed}"

run_all(SEEDS)

2 GPU(s) -> up to 2 seeds in parallel
[seed 43 | gpu 1] 
[seed 42 | gpu 0] 
[seed 42 | gpu 0] [transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
[seed 42 | gpu 0] Key                        | Status     | 
[seed 43 | gpu 1] [transformers] XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
[seed 43 | gpu 1] Key                        | Status     | 
[seed 43 | gpu 1] ---------------------------+------------+-
[seed 42 | gpu 0] ---------------------------+------------+-
[seed 42 | gpu 0] lm_head.dense.weight       | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.bias               | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.decoder.weight     | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.dense.weight       | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.layer_norm.weight  | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.decoder.bias       | UNEXPECTED | 
[seed 43 | gpu 1] lm_head.decoder.bias       | UNEXPECTED | 
[seed 42 | gpu 0] lm_head.l

## 4 · Results across seeds

Macro-F1 is the primary metric; weighted-F1 is the official SemEval-2023 metric. `clean` excludes tweets that duplicate a training tweet; `no_tones` / `no_diacritics` score the same tweets with tone marks / all diacritics removed.

In [4]:
import pandas as pd

rows = [r for s in SEEDS for r in json.load(open(f"{OUT}/seed{s}/metrics.json"))]
df = pd.DataFrame(rows)
summary = (df.groupby(["split", "subset", "eval_diacritics"])[["macro_f1", "weighted_f1", "accuracy"]]
             .agg(["mean", "std"]).round(4))
summary

macro_f1         weighted_f1         accuracy  \
                                 mean     std        mean     std     mean   
split subset eval_diacritics                                                 
dev   all    no_diacritics     0.7428  0.0124      0.7627  0.0101   0.7636   
             no_tones          0.7100  0.0151      0.7313  0.0145   0.7319   
             original          0.6532  0.0287      0.6760  0.0280   0.6789   
      clean  no_diacritics     0.7135  0.0127      0.7351  0.0099   0.7362   
             no_tones          0.6809  0.0164      0.7034  0.0157   0.7041   
             original          0.6260  0.0307      0.6489  0.0310   0.6526   

                                      
                                 std  
split subset eval_diacritics          
dev   all    no_diacritics    0.0113  
             no_tones         0.0148  
             original         0.0244  
      clean  no_diacritics    0.0110  
             no_tones         0.0154  
             original         0.0256

In [5]:
per_seed = df[(df.subset == "clean") & (df.eval_diacritics == "original")][
    ["split", "seed", "macro_f1", "weighted_f1", "f1_negative", "f1_neutral", "f1_positive"]]
per_seed

,split,seed,macro_f1,weighted_f1,f1_negative,f1_neutral,f1_positive
1,dev,42,0.6457,0.6698,0.5173,0.6915,0.7284
7,dev,43,0.5906,0.6133,0.4603,0.6663,0.6452
13,dev,44,0.6418,0.6635,0.5278,0.6772,0.7205
